# Playground S6E10 — Airline Passenger Satisfaction

**Task:** predict the probability that a passenger is satisfied. **Metric:** ROC AUC.

| Step | CV AUC |
|---|---|
| LightGBM, raw features | 0.95882 |
| + numeric columns as categorical copies | 0.96002 |
| CatBoost, every column categorical | 0.96056 |
| XGBoost + v2 features (route encodings, original-data model, route profiles) | 0.96108 |
| LightGBM + v2 features | 0.96105 |
| **Logistic-regression stack** | **0.96136** (public LB 0.96102) |

Sections: 1. Setup · 2. Data · 3. Quick EDA · 4. Features · 5. Cross-validation · 6. Models · 7. Stack & submission

## 1. Setup
Works locally (project folder) and on Kaggle (`/kaggle/input`).

In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, Pool
warnings.filterwarnings("ignore")

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    COMP = Path("/kaggle/input/competitions/playground-series-s6e10")
    ORIG = next(Path("/kaggle/input").rglob("airline-passenger-satisfaction*"))
else:
    COMP = Path("../data")
    ORIG = Path("../data/original")

TARGET, SEED, N_FOLDS = "satisfaction", 42, 5
try:
    import torch; GPU = torch.cuda.is_available()
except ImportError:
    GPU = True   # set False if you have no NVIDIA GPU
FAST = False     # True = quick dry run on a 50k-row sample

## 2. Data
Competition data (synthetic) + the original real dataset it was generated from (~130k rows).

In [ ]:
train = pd.read_csv(COMP / "train.csv")
test = pd.read_csv(COMP / "test.csv")
orig = pd.concat([pd.read_csv(ORIG / f) for f in ("train.csv", "test.csv")], ignore_index=True)
orig = orig.drop(columns=["Unnamed: 0", "id", "Inflight service"])
orig[TARGET] = (orig[TARGET] == "satisfied").astype(int)
train[TARGET] = train[TARGET].astype(int)
if FAST:
    train = train.sample(50_000, random_state=0).reset_index(drop=True)

CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
RATING_COLS = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
               "Gate location", "Food and drink", "Online boarding", "Seat comfort",
               "Inflight entertainment", "On-board service", "Leg room service",
               "Baggage handling", "Checkin service", "Cleanliness"]
NUM_COLS = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]
BASE = NUM_COLS + RATING_COLS + CAT_COLS
FD = "Flight Distance"
print(train.shape, test.shape, orig.shape, "| satisfied rate:", train[TARGET].mean().round(4))

## 3. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["Online boarding", "Inflight wifi service", "Class"]):
    train.groupby(c)[TARGET].mean().plot.bar(ax=ax, color="#2563eb")
    ax.set_title(f"Satisfied rate by {c}"); ax.set_ylim(0, 1); ax.set_ylabel("")
plt.tight_layout(); plt.show()

print("Distinct Flight Distance values:", train[FD].nunique(),
      "-> each exact distance behaves like a route id")

## 4. Features
* **Engineered:** rating sum / std, number of 0, 5 and low ratings, delay total / difference.
* **og_lgb:** logit of a LightGBM trained *only on the original dataset* (external data, no leakage).
* **cnt_\*:** frequency of each numeric value in train+test.
* **fdm_\*, fd_cnt:** route profile — mean of every column per Flight Distance (no target used).
* **Target-encoding keys** (encoded inside each CV fold): FD, Age, FD×Age, FD×Class, FD×Travel, Age×Class×Travel×Customer.

In [ ]:
def add_features(df):
    df = df.copy()
    r = df[RATING_COLS]
    df["rating_sum"], df["rating_std"] = r.sum(axis=1), r.std(axis=1)
    df["n_zero"], df["n_five"], df["n_low"] = (r == 0).sum(axis=1), (r == 5).sum(axis=1), (r <= 2).sum(axis=1)
    df["delay_total"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]
    df["delay_diff"] = df["Arrival Delay in Minutes"] - df["Departure Delay in Minutes"]
    df["arr_na"] = df["Arrival Delay in Minutes"].isna().astype(int)
    for c in CAT_COLS:
        df[c] = df[c].astype("category")
    return df

ENG = ["rating_sum", "rating_std", "n_zero", "n_five", "n_low", "delay_total", "delay_diff", "arr_na"]
train, test, orig = add_features(train), add_features(test), add_features(orig)

full = pd.concat([train[BASE], test[BASE]], ignore_index=True)
extra = pd.DataFrame(index=full.index)

# model trained on the original data only
og = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, num_leaves=63, subsample=0.8, subsample_freq=1,
                        colsample_bytree=0.5, verbose=-1, n_jobs=-1, random_state=0).fit(orig[BASE], orig[TARGET])
p = np.clip(og.predict_proba(full)[:, 1], 1e-6, 1 - 1e-6)
extra["og_lgb"] = np.log(p / (1 - p)).astype("float32")

for c in NUM_COLS:
    extra["cnt_" + c] = full[c].map(full[c].value_counts()).fillna(0).astype("float32")

coded = full.copy()
for c in CAT_COLS:
    coded[c] = coded[c].cat.codes
grp = coded.groupby(FD)
for c in BASE:
    if c != FD:
        extra["fdm_" + c] = grp[c].transform("mean").astype("float32")
extra["fd_cnt"] = grp["Age"].transform("size").astype("float32")

TE_KEYS = [[FD], ["Age"], [FD, "Age"], [FD, "Class"], [FD, "Type of Travel"],
           ["Age", "Class", "Type of Travel", "Customer Type"]]
keys = pd.DataFrame(index=full.index)
for cols in TE_KEYS:
    s = full[cols[0]].astype(str)
    for c in cols[1:]:
        s = s + "|" + full[c].astype(str)
    keys["te_" + "|".join(cols)] = pd.factorize(s)[0]

n = len(train)
X = pd.concat([train[BASE + ENG], extra.iloc[:n].reset_index(drop=True)], axis=1)
X_test = pd.concat([test[BASE + ENG].reset_index(drop=True), extra.iloc[n:].reset_index(drop=True)], axis=1)
K, K_test = keys.iloc[:n].reset_index(drop=True), keys.iloc[n:].reset_index(drop=True)
y = train[TARGET].values
print("features:", X.shape[1] + K.shape[1])

## 5. Cross-validation
Target encoding is fitted inside each training fold with an inner 5-fold cross-fit, so no row ever sees its own label.

In [ ]:
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
OOF, TEST = {}, {}

def run_cv(name, fit_predict):
    oof, tst, t0 = np.zeros(len(X)), np.zeros(len(X_test)), time.time()
    for k, (a, b) in enumerate(FOLDS):
        enc = TargetEncoder(target_type="binary", smooth="auto", cv=5, shuffle=True, random_state=0)
        Xa, Xb, Xt = X.iloc[a].copy(), X.iloc[b].copy(), X_test.copy()
        Xa[K.columns] = enc.fit_transform(K.iloc[a], y[a])
        Xb[K.columns] = enc.transform(K.iloc[b])
        Xt[K.columns] = enc.transform(K_test)
        pb, pt = fit_predict(Xa, y[a], Xb, y[b], Xt)
        oof[b], tst = pb, tst + pt / N_FOLDS
        print(f"  fold {k}: {roc_auc_score(y[b], pb):.5f}  ({time.time() - t0:.0f}s)")
    OOF[name], TEST[name] = oof, tst
    print(f"{name}: CV AUC {roc_auc_score(y, oof):.5f}")

## 6. Models

In [ ]:
def fit_xgb(Xa, ya, Xb, yb, Xt):
    m = xgb.XGBClassifier(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=5,
                          subsample=0.8, colsample_bytree=0.5, reg_lambda=3.0, tree_method="hist",
                          device="cuda" if GPU else "cpu", enable_categorical=True, max_cat_to_onehot=4,
                          eval_metric="auc", early_stopping_rounds=300, random_state=0)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]

run_cv("xgb", fit_xgb)

In [ ]:
def fit_lgb(Xa, ya, Xb, yb, Xt):
    m = lgb.LGBMClassifier(n_estimators=10000, learning_rate=0.02, num_leaves=63, min_child_samples=40,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=3.0,
                           max_cat_to_onehot=4, cat_smooth=20, cat_l2=10, verbose=-1, random_state=0, n_jobs=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric="auc", callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]

run_cv("lgb", fit_lgb)

In [ ]:
# CatBoost: every raw column also as a categorical string -> captures synthetic-data value artifacts.
# Slowest model (~10 min/fold on a small GPU).
def fit_cat(Xa, ya, Xb, yb, Xt):
    def prep(D):
        D = D.copy()
        for c in CAT_COLS:
            D[c] = D[c].astype(str)
        for c in NUM_COLS + RATING_COLS:
            D[c + "_c"] = D[c].fillna(-1).astype(int).astype(str)
        return D
    Xa, Xb, Xt = prep(Xa), prep(Xb), prep(Xt)
    cats = CAT_COLS + [c for c in Xa.columns if c.endswith("_c")]
    m = CatBoostClassifier(iterations=10000, learning_rate=0.08, depth=7, l2_leaf_reg=3, eval_metric="AUC",
                           task_type="GPU" if GPU else "CPU", metric_period=50, early_stopping_rounds=300,
                           random_seed=0, verbose=0)
    m.fit(Pool(Xa, ya, cat_features=cats), eval_set=Pool(Xb, yb, cat_features=cats))
    return (m.predict_proba(Pool(Xb, cat_features=cats))[:, 1],
            m.predict_proba(Pool(Xt, cat_features=cats))[:, 1])

run_cv("cat", fit_cat)

## 7. Stack & submission
Logistic regression on the models' OOF logits, scored with the same folds.

In [ ]:
logit = lambda a: np.log(np.clip(a, 1e-6, 1 - 1e-6) / (1 - np.clip(a, 1e-6, 1 - 1e-6)))
names = list(OOF)
S, S_test = np.column_stack([logit(OOF[k]) for k in names]), np.column_stack([logit(TEST[k]) for k in names])

stack_oof = np.zeros(len(y))
for a, b in FOLDS:
    stack_oof[b] = LogisticRegression(max_iter=1000).fit(S[a], y[a]).decision_function(S[b])
for k in names:
    print(f"{k:6s} {roc_auc_score(y, OOF[k]):.5f}")
print(f"stack  {roc_auc_score(y, stack_oof):.5f}")

lr = LogisticRegression(max_iter=1000).fit(S, y)
sub = pd.DataFrame({"id": test["id"], TARGET: lr.predict_proba(S_test)[:, 1]})
sub.to_csv("submission.csv", index=False)
sub.head()